In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer
import chromadb
from chromadb.config import Settings

import json
import shutil
from pathlib import Path
from typing import List, Any

In [43]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [24]:
documents = []

for file_path in Path("../data/").glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        articles = json.load(f)
        
    
    for article in articles["data"]["results"]:
        documents.append(
            Document(
                page_content=article["content"],
                metadata={
                    "id":           article["id"],
                    "title":        article["title"],
                    "source":       article["source"],
                    "link":         str(article["link"]),
                    "publish_date": str(article["publish_date"]),
                    "images":       str(article.get("images")),
                }
            )
        )

print(f"Loaded {len(documents)} articles from {len(list(Path('../data/').glob('*.json')))} files")

Loaded 440 articles from 11 files


In [28]:
print(documents[0].metadata)
print(documents[0].page_content[:500])

{'id': 'c0a315ff3558239f19d0482261befcfa6fff9359b5e47eeaff1630ef5d8723b5', 'title': 'Iceland Joins Switzerland, Turkey, Ireland, Denmark (Greenland), Colombia, and Italy as the Must-Visit Destinations for US Travelers in 2026, Offering Stunning Landscapes and Unique Adventures - Travel And Tour World', 'source': 'feeds.feedburner.com', 'link': 'https://www.travelandtourworld.com/news/article/iceland-joins-switzerland-turkey-ireland-denmark-greenland-colombia-and-italy-as-the-must-visit-destinations-for-us-travelers-in-2026-offering-stunning-landscapes-and-unique-adventures/', 'publish_date': '2025-12-20T08:00:00+08:00', 'images': "['https://www.travelandtourworld.com/wp-content/uploads/2025/12/iceland-joins-switzerland-turkey-ireland_j_GaAnXzQI2f3ZNahd3b_Q_EvSbLFoeR8S583HtKh2ldg.jpg']"}
Home»America Travel News» Iceland Joins Switzerland, Turkey, Ireland, Denmark (Greenland), Colombia, and Italy as the Must-Visit Destinations for US Travelers in 2026, Offering Stunning Landscapes and U

In [ ]:
class EmbeddingDocument:
    def __init__(
        self,
        persist_dir: str = "../data/chroma_db",
        collection_name: str = "embedded_articles",
        embedding_model: str = "text-embedding-3-small",
        chat_model: str = "gpt-4o-mini",
        capacity: int = 256,
    ):
        self.persist_dir = persist_dir
        self.collection_name = collection_name
        self.capacity = capacity
        tokenizer = Tokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        self.splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, capacity=capacity)
        self.embedding = OpenAIEmbeddings(model=embedding_model)
        self.llm = ChatOpenAI(model=chat_model, temperature=0)
        self.vectorstore = None
        print(f"[INFO] EmbeddingDocument ready (embedding={embedding_model}, chat={chat_model}, capacity={capacity})")

    def _chroma_client(self):
        """Create a ChromaDB client with writable settings."""
        return chromadb.PersistentClient(
            path=self.persist_dir,
            settings=Settings(anonymized_telemetry=False, allow_reset=True),
        )

    def chunk_documents(self, documents: List[Any]) -> List[Document]:
        chunked_documents = []
        for doc in documents:
            chunks = self.splitter.chunks(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunked_documents.append(
                    Document(
                        page_content=chunk,
                        metadata={
                            **doc.metadata,
                            "id":          f"{doc.metadata['id']}_chunk_{i}",
                            "chunk_index": i,
                            "parent_id":   doc.metadata.get("id"),
                        }
                    )
                )
        print(f"[INFO] Split {len(documents)} documents into {len(chunked_documents)} chunks.")
        return chunked_documents

    def embed_documents(self, documents: List[Any]):
        """Chunk, embed, and persist documents into Chroma. Overwrites if DB already exists."""
        persist_path = Path(self.persist_dir)
        if persist_path.exists():
            print(f"[INFO] Existing database found at '{self.persist_dir}', overwriting...")
            shutil.rmtree(persist_path)
        persist_path.mkdir(parents=True, exist_ok=True)

        chunks = self.chunk_documents(documents)
        print(f"[INFO] Embedding and storing {len(chunks)} chunks...")
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embedding,
            client=self._chroma_client(),
            collection_name=self.collection_name,
        )
        print(f"[INFO] Stored {self.vectorstore._collection.count()} chunks in Chroma at '{self.persist_dir}'")

    def load(self):
        """Load an existing Chroma store from disk."""
        self.vectorstore = Chroma(
            client=self._chroma_client(),
            embedding_function=self.embedding,
            collection_name=self.collection_name,
        )
        print(f"[INFO] Loaded {self.vectorstore._collection.count()} chunks from '{self.persist_dir}'")

    def query(self, query_text: str, top_k: int = 5) -> List[Document]:
        """Retrieve top-k most relevant chunks for a query."""
        if self.vectorstore is None:
            raise RuntimeError("No vectorstore loaded. Call embed_documents() or load() first.")
        print(f"[INFO] Querying for: '{query_text}'")
        return self.vectorstore.similarity_search(query_text, k=top_k)

    def generate_response(self, query_text: str, top_k: int = 20) -> str:
        """Retrieve top_k chunks and generate an answer using the LLM."""
        chunks = self.query(query_text, top_k=top_k)

        if not chunks:
            return "Sorry, I couldn't find any relevant articles."

        context = "\n\n".join(
            f"[Source: {doc.metadata.get('title', 'Unknown')} | {doc.metadata.get('publish_date', '')}]\n{doc.page_content}"
            for doc in chunks
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question using only the provided context. "
                    "If the context does not contain enough information, say so. "
                    "Cite the article titles where relevant.\n\n"
                    f"Context:\n{context}"
                )
            },
            {
                "role": "user",
                "content": query_text
            }
        ]

        response = self.llm.invoke(messages)
        print(f"[INFO] Generated response from {len(chunks)} chunks.")
        return response.content

In [ ]:
pipeline = EmbeddingDocument()
final_chunked = pipeline.chunk_documents(documents)

# Print chunks from the first article
first_id = final_chunked[0].metadata["parent_id"]
first_article_chunks = [c for c in final_chunked if c.metadata["parent_id"] == first_id]

for chunk in first_article_chunks:
    print(f"--- Chunk {chunk.metadata['chunk_index']} ---")
    print(chunk.page_content)
    print()

In [ ]:
# Embed and persist (overwrites existing DB if present)
pipeline = EmbeddingDocument()
pipeline.embed_documents(documents)

# Generate a response from top 20 chunks
response = pipeline.generate_response("best destinations for US travelers in 2026", top_k=20)
print(response)